# HH2 - STEP 1: collect and structure (merge) datasets

### Import libraries 

In [4]:
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

## Load and view data 

### Indoor temperature

In [7]:
# Get the database from the DataFoundry link
df_indoor = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/Yy9PQlp3clNidmNyc0pYS1JBV1NlQ1JpbGNKWHBIQlVVMjlwQW9nOFY5UT0=", low_memory=False)
df_indoor = df_indoor[(df_indoor.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_indoor = df_indoor.drop(["Unnamed: 7", "device_id", "activity", "participant", "sender", "id", "recipient", "pp1", "pp2", "pp3"], axis='columns')
df_indoor = df_indoor.rename(columns={"Temperature": "Temperature_indoor"}, errors="raise")
df_indoor['ts'] = pd.to_datetime(df_indoor['ts']) ## Turn timestamp into datetime dtype
df_indoor['Temperature_indoor'] = np.round(df_indoor['Temperature_indoor'] * 10) / 10 ## round temperature to 1 decimal
df_indoor = df_indoor.dropna() ## drop rows with empty (NA) cells
df_indoor = df_indoor.drop_duplicates() ## Drop duplicate rowsindex_list= df_indoor2.Timestamp[(df_indoor2.Timestamp >= "2024-08-08 16:00:00") & (df_indoor2.Timestamp <= "2024-08-08 19:20:00")].index.tolist(
df_indoor['hr'] = df_indoor['ts'].dt.hour
df_indoor['date'] = df_indoor['ts'].dt.date
df_indoor = df_indoor.groupby(['date', 'hr']).first().reset_index()
df_indoor['ts'] = df_indoor["ts"].dt.round('h')  #Round the datestamp column to hours
df_indoor = df_indoor.drop(["hr", "date"], axis='columns')

display(df_indoor.tail())
display(df_indoor.describe())

,ts,Temperature_indoor
791,2024-09-30 10:00:00,19.4
792,2024-09-30 11:00:00,19.3
793,2024-09-30 12:00:00,19.3
794,2024-09-30 13:00:00,19.3
795,2024-09-30 14:00:00,19.3


,ts,Temperature_indoor
count,796,796.000000
mean,2024-09-14 00:29:28.341708544,23.439196
min,2024-08-28 09:00:00,19.300000
25%,2024-09-05 17:45:00,22.000000
50%,2024-09-14 00:30:00,23.300000
75%,2024-09-22 07:15:00,24.700000
max,2024-09-30 14:00:00,28.600000
std,NaN,1.943529


### API Outdoor temperature

In [9]:
# Get the database from the DataFoundry link
df_API = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/N0FsN2loZlVYMWhBaE0rd0l5T2NadzR3YTNBcnovQlJ0SG13dHMxL0U1RT0=")
df_API = df_API[(df_API.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_API = df_API.drop(["activity", "device_id", "sender", "participant", "Unnamed: 7", "id", "recipient", "pp1", "pp2", "pp3"], axis='columns')
df_API['ts'] = pd.to_datetime(df_API['ts']) ## Turn timestamp into datetime dtype
# df_API['Temperature_API'] = np.round(df_API['Temperature_API'] * 10) / 10 ## round temperature to 1 decimal
df_API['hr'] = df_API['ts'].dt.hour
df_API['date'] = df_API['ts'].dt.date
df_API = df_API.groupby(['date', 'hr']).first().reset_index()
df_API['ts'] = df_API["ts"].dt.round('h')  #Round the datestamp column to hours
df_API = df_API.drop(["hr", "date"], axis='columns')
df_API.head()

,ts,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,weather_description,weather_main
0,2024-08-28 09:00:00,16.74,18.27,15.43,broken clouds,Clouds
1,2024-08-28 09:00:00,18.03,18.83,17.11,broken clouds,Clouds
2,2024-08-28 10:00:00,21.85,22.72,20.43,broken clouds,Clouds
3,2024-08-28 14:00:00,28.18,29.92,26.92,few clouds,Clouds
4,2024-08-28 15:00:00,28.70,29.92,27.65,few clouds,Clouds


### Door (indoor) state

In [11]:
# Get the database from the DataFoundry link
df_door = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/S0lMV3BUZ3NjZ2lQVUJUVHlJREhSM2NxNkFGT0VscHFHY0xjdXZVTS9SST0=")
df_door = df_door[(df_door.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_door = df_door.drop(["window", "Unnamed: 7", "device_id", "activity", "participant", "sender", "id", "recipient", "pp1", "pp2", "pp3"], axis='columns')
df_door = df_door.rename(columns={"reed sensor": "door"}, errors="raise")
df_door['ts'] = pd.to_datetime(df_door['ts']) ## Turn timestamp into datetime dtype
df_door = df_door.dropna() ## drop rows with empty (NA) cells
df_door = df_door.drop_duplicates() ## Drop duplicate rows
df_door['hr'] = df_door['ts'].dt.hour
df_door['date'] = df_door['ts'].dt.date
df_door = df_door.groupby(['date', 'hr']).agg(lambda x: x.mode().max()).reset_index() #get the mode for each hour since its possible they only opened it for a few mins at the beginning of the hour
# df_door = df_door.groupby(['date', 'hr']).agg(pd.Series.mode).reset_index()
df_door['ts'] = df_door["ts"].dt.round('h')  #Round the datestamp column to hours
df_door = df_door.drop(["hr", "date"], axis='columns')
df_door.tail()

,ts,door
790,2024-09-30 11:00:00,0.0
791,2024-09-30 12:00:00,0.0
792,2024-09-30 13:00:00,0.0
793,2024-09-30 14:00:00,0.0
794,2024-09-30 15:00:00,0.0


### Window state

In [13]:
# Get the database from the DataFoundry link
df_window = pd.read_csv (r"https://data.id.tue.nl/datasets/downloadPublic/QWM2TVQ4OE9KRWF1UU1tRUxzbXJFS1IxM3hhMVZORnNlSlMvRmVyZUFsdz0=", low_memory=False)
df_window = df_window[(df_window.participant == "H2")] # select the correct participant

# Clean up the dataframe
df_window = df_window.drop(["Unnamed: 7", "device_id", "id", "participant", "recipient", "pp1", "pp2", "pp3", "activity", "light", "participant", "curtain"], axis='columns')

# get the left window sensor values
df_window_left = df_window.loc[(df_window['sender'] == "HH2_window_1") | (df_window['sender'] == "HH2_window_1_V2")].copy()
df_window_left["curtain_left"] = np.where(df_window_left.loc[:,"distance sensor"] > 50, 1, 0) # set curtain state (0= closed; 1 = open)
# df_window_left = df_window_left.rename(columns={"light sensor": "light_left"}, errors="raise")
df_window_left["shade_left"] = np.where(df_window_left.loc[:,"light sensor"] < 2000, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_left = df_window_left.rename(columns={"light sensor": "light_left"}, errors="raise")
df_window_left = df_window_left.rename(columns={"distance sensor": "distance_left"}, errors="raise")
df_window_left = df_window_left.drop(["sender", "window", "reed sensor"], axis='columns')
df_window_left['ts'] = pd.to_datetime(df_window_left['ts']) #Turn timestamp into datetime dtype
df_window_left = df_window_left.drop_duplicates() #Drop duplicate rows
# df_window_left.resample('10min', on='ts').last() #get one value per 10 minutes
df_window_left['hr'] = df_window_left['ts'].dt.hour
df_window_left['date'] = df_window_left['ts'].dt.date
df_window_left = df_window_left.groupby(['date', 'hr']).agg(lambda x: x.mode().max()).reset_index() #get the mode for each hour since its possible they only opened it for a few mins at the beginning of the hour
df_window_left['ts'] = df_window_left["ts"].dt.round('h')  #Round the datestamp column to hours
df_window_left = df_window_left.drop(["hr", "date"], axis='columns')

# # get the right window sensor values
df_window_right = df_window.loc[df_window['sender'] == "HH2_window_2"].copy()
df_window_right["curtain_right"] = np.where(df_window_right.loc[:,"distance sensor"] > 50, 1, 0) # set curtain state (0= closed; 1 = open)
# df_window_right = df_window_right.rename(columns={"light sensor": "light_right"}, errors="raise")
df_window_right["shade_right"] = np.where(df_window_right.loc[:,"light sensor"] < 2000, 1, 0) # set shade (0= no shade; 1 = yes shade)
df_window_right = df_window_right.rename(columns={"light sensor": "light_right"}, errors="raise")
df_window_right = df_window_right.rename(columns={"distance sensor": "distance_right"}, errors="raise")
df_window_right = df_window_right.drop(["sender", "window", "reed sensor"], axis='columns')
df_window_right['ts'] = pd.to_datetime(df_window_right['ts']) #Turn timestamp into datetime dtype
df_window_right = df_window_right.drop_duplicates() #Drop duplicate rows
# df_window_right.resample('10min', on='ts').last() #get one value per 10 minutes
# df_window_right['ts'] = df_window_right["ts"].dt.round('h')  #Round the datestamp column to hours
df_window_right['hr'] = df_window_right['ts'].dt.hour
df_window_right['date'] = df_window_right['ts'].dt.date
df_window_right = df_window_right.groupby(['date', 'hr']).agg(lambda x: x.mode().max()).reset_index() #get the mode for each hour since its possible they only opened it for a few mins at the beginning of the hour
df_window_right['ts'] = df_window_right["ts"].dt.round('h')  #Round the datestamp column to hours
df_window_right = df_window_right.drop(["hr", "date"], axis='columns')


# # get the right window sensor values
df_window_door = df_window.loc[df_window['sender'] == "HH2_windowdoor_1"].copy()
df_window_door = df_window_door.rename(columns={"reed sensor": "window_door"}, errors="raise")
df_window_door = df_window_door.drop(["sender", "light sensor", "window", "distance sensor"], axis='columns')
df_window_door['ts'] = pd.to_datetime(df_window_door['ts']) #Turn timestamp into datetime dtype
df_window_door = df_window_door.drop_duplicates() #Drop duplicate rows
# df_window_door.resample('10min', on='ts').last() #get one value per 10 minutes
df_window_door['hr'] = df_window_door['ts'].dt.hour
df_window_door['date'] = df_window_door['ts'].dt.date
df_window_door = df_window_door.groupby(['date', 'hr']).agg(lambda x: x.mode().max()).reset_index() #get the mode for each hour since its possible they only opened it for a few mins at the beginning of the hour
df_window_door['ts'] = df_window_door["ts"].dt.round('h')  #Round the datestamp column to hours
df_window_door = df_window_door.drop(["hr", "date"], axis='columns')

### merge the window dataframes on the timestamps for the last 72 hours
from functools import reduce
# Create a list of DataFrames to merge
dataframes_to_merge = [
    df_window_door,
    df_window_left,
    df_window_right,
]

# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))

# Use reduce to merge all DataFrames in one go
df_window = reduce(merge_asof, dataframes_to_merge)
df_window.tail()
df_window.describe()
# df_window = df_window.drop_duplicates(subset=['ts']) #Drop duplicate rows

,ts,window_door,distance_left,light_left,curtain_left,shade_left,distance_right,light_right,curtain_right,shade_right
count,762,762.000000,641.000000,641.000000,641.000000,641.000000,762.000000,762.000000,762.000000,762.000000
mean,2024-09-14 10:35:25.984251904,0.240157,293.553822,1533.190328,0.511700,0.613105,274.024934,1458.044619,0.524934,0.639108
min,2024-08-28 15:00:00,0.000000,0.000000,0.000000,0.000000,0.000000,8.000000,0.000000,0.000000,0.000000
25%,2024-09-05 14:00:00,0.000000,12.000000,0.000000,0.000000,0.000000,14.000000,0.000000,0.000000,0.000000
50%,2024-09-14 18:30:00,0.000000,49.000000,0.000000,1.000000,1.000000,53.000000,792.000000,1.000000,1.000000
75%,2024-09-22 16:45:00,0.000000,799.000000,3247.000000,1.000000,1.000000,462.500000,3067.000000,1.000000,1.000000
max,2024-09-30 15:00:00,1.000000,803.000000,4095.000000,1.000000,1.000000,801.000000,4001.000000,1.000000,1.000000
std,NaN,0.427460,335.845594,1666.958094,0.500253,0.487420,309.677800,1569.811531,0.499706,0.480575


## MERGE ON TIMESTAMP

In [16]:
### merge the window dataframes on the timestamps for the last 72 hours
from functools import reduce
# Create a list of DataFrames to merge
dataframes_to_merge = [
    df_indoor,
    df_API,
    df_window,
    df_door,
]

# Function to merge two DataFrames
def merge_asof(df1, df2):
    return pd.merge_asof(df1.sort_values('ts'), df2.sort_values('ts'), on='ts', tolerance=pd.Timedelta('120 min'))

# Use reduce to merge all DataFrames in one go
df_hourly = reduce(merge_asof, dataframes_to_merge)
df_hourly.tail()
# df_hourly.describe()

,ts,Temperature_indoor,Temperature_API,Temperature_API_MAX,Temperature_API_MIN,weather_description,weather_main,window_door,distance_left,light_left,curtain_left,shade_left,distance_right,light_right,curtain_right,shade_right,door
791,2024-09-30 10:00:00,19.4,11.25,12.13,10.37,light rain,Rain,0.0,NaN,NaN,NaN,NaN,9.0,1692.0,0.0,1.0,0.0
792,2024-09-30 11:00:00,19.3,12.00,12.97,11.49,light rain,Rain,0.0,NaN,NaN,NaN,NaN,9.0,1958.0,0.0,1.0,0.0
793,2024-09-30 12:00:00,19.3,12.19,13.22,11.50,light rain,Rain,0.0,NaN,NaN,NaN,NaN,9.0,2064.0,0.0,1.0,0.0
794,2024-09-30 13:00:00,19.3,12.63,13.80,11.55,light rain,Rain,0.0,NaN,NaN,NaN,NaN,9.0,2391.0,0.0,0.0,0.0
795,2024-09-30 14:00:00,19.3,13.28,14.91,11.90,moderate rain,Rain,0.0,NaN,NaN,NaN,NaN,9.0,2105.0,0.0,1.0,0.0


In [17]:
df_hourly['hr'] = df_hourly['ts'].dt.hour
df_hourly['date'] = df_hourly['ts'].dt.date

## SAVE AS CSV

In [31]:
df_hourly.to_csv(r"C:\Users\20204113\OneDrive - TU Eindhoven\2_Research\Year4_CoolAI\Jupyter_notebooks\H2\DATA (backups)\1_hourly.csv", index = None)